### Open AI Embeddings

In [25]:
import os 
from dotenv import load_dotenv
load_dotenv()


True

In [26]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [27]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


In [28]:
#Single text embeddings
single_text = "Langchain and RAG are amazing framework and projects to work on"
single_embeddings = embeddings.embed_query(single_text)
print(len(single_embeddings))
print(single_embeddings)

1536
[-0.045623779296875, -0.023529052734375, 0.00952911376953125, -0.0022735595703125, 0.01421356201171875, -0.029693603515625, -0.0002288818359375, -0.0036945343017578125, -0.01126861572265625, -0.0310821533203125, 0.0157012939453125, 0.0096588134765625, -0.039886474609375, 0.04425048828125, 0.005619049072265625, -0.0144195556640625, -6.937980651855469e-05, -0.053863525390625, 0.02850341796875, 0.0750732421875, -0.0214385986328125, -0.002849578857421875, -0.00853729248046875, 0.04083251953125, -0.0179443359375, -0.01340484619140625, -0.0034008026123046875, 0.0699462890625, 0.01262664794921875, -0.0269317626953125, 0.0015764236450195312, -0.035675048828125, 0.005939483642578125, 0.0238494873046875, 0.00667572021484375, 0.0269012451171875, -0.00228118896484375, -0.01128387451171875, -0.01265716552734375, 0.020111083984375, 0.0208282470703125, 0.0302276611328125, 0.007015228271484375, 0.00968170166015625, 0.01529693603515625, -0.01526641845703125, -0.04351806640625, 0.026611328125, 0.00

In [29]:
print(" 📄 Single text embedding")
print(f"Input {single_text}")
print(f"Output: Vector of {len(single_embeddings)} dimension")
print(f"Sample value: {single_embeddings[:5]}")

 📄 Single text embedding
Input Langchain and RAG are amazing framework and projects to work on
Output: Vector of 1536 dimension
Sample value: [-0.045623779296875, -0.023529052734375, 0.00952911376953125, -0.0022735595703125, 0.01421356201171875]


In [30]:
#Example2: Multiple text as once
multiple_text = [
    "The cat sat on the mat",
    "A feline rested on the rug",
    "The dog played in the yard",
    "I love programming in python",
    "Python is my favorite programming language"
]


In [31]:
multiple_embeddings = embeddings.embed_documents(multiple_text)

### Cosine similarity with OpenAI Embeddings

In [33]:
#Ex 1: finding similiar sentences
sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",
    "The dog played in the yard",
    "I love programming in python",
    "Python is my favorite programming language"
]

In [34]:
import numpy as np
def cosine_similarity(vec1, vec2):
    """
    cosine similarity measures the angle between two vectors.
    result close to 1: very similar
    result close to 0: not related
    result close to -1: opposite meaning
    """
    
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    return dot_product/(norm_a * norm_b)

In [35]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x0000020D4A2A7F80>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x0000020D4A6E9E50>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [47]:
sentences_embeddings = embeddings.embed_documents(sentences)


In [37]:
#calculate the similarity vbetween al pairs

for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        similarity = cosine_similarity(sentences_embeddings[i], sentences_embeddings[j])

        print(f"'{sentences[i]}' vs '{sentences[j]}'")
        print(f"Similarity: {similarity:.3f}\n")

'The cat sat on the mat' vs 'A feline rested on the rug'
Similarity: 0.656

'The cat sat on the mat' vs 'The dog played in the yard'
Similarity: 0.324

'The cat sat on the mat' vs 'I love programming in python'
Similarity: 0.096

'The cat sat on the mat' vs 'Python is my favorite programming language'
Similarity: 0.120

'A feline rested on the rug' vs 'The dog played in the yard'
Similarity: 0.296

'A feline rested on the rug' vs 'I love programming in python'
Similarity: 0.058

'A feline rested on the rug' vs 'Python is my favorite programming language'
Similarity: 0.103

'The dog played in the yard' vs 'I love programming in python'
Similarity: 0.128

'The dog played in the yard' vs 'Python is my favorite programming language'
Similarity: 0.085

'I love programming in python' vs 'Python is my favorite programming language'
Similarity: 0.734



In [38]:
# test sementic search
documents= [
    "langchain is  a frame work for devolapinf=g aio app",
    "Python is a high level language",
    "Machin learning is a subset of AI",
    "Embedding convert text into numrical vectors"
]
query = "What is langchain?"

In [43]:


def semantic_search(query, documents, embeddings_models, top_k=3):
    """simple semantic search implemention"""

    #embed query and docunment

    query_embedding = embeddings_models.embed_query(query)
    doc_embeddings = embeddings_models.embed_documents(documents)

    #calcu;ate the similarity score

    similarities = []

    for i, doc_emb in enumerate(doc_embeddings):
        similarity = cosine_similarity(query_embedding, doc_emb)
        similarities.append((similarity, documents[i]))

        ##sort by similarity
    similarities.sort(reverse=True)
    return similarities[:top_k]

In [44]:
result = semantic_search(query=query, documents=documents, embeddings_models=embeddings)
result

[(np.float64(0.5943153732496111),
  'langchain is  a frame work for devolapinf=g aio app'),
 (np.float64(0.20051174809092542), 'Python is a high level language'),
 (np.float64(0.15970014646424482), 'Machin learning is a subset of AI')]

In [45]:
print(f"\n 🔍 semantic search results for: '{query}'")
for score, doc in result:
    print(f"Score: {score:.3f} | {doc}")


 🔍 semantic search results for: 'What is langchain?'
Score: 0.594 | langchain is  a frame work for devolapinf=g aio app
Score: 0.201 | Python is a high level language
Score: 0.160 | Machin learning is a subset of AI


In [46]:
query = "ehat is embedding"
results = semantic_search(query, documents, embeddings)
results

[(np.float64(0.27676995829424744),
  'Embedding convert text into numrical vectors'),
 (np.float64(0.2613594696128452),
  'langchain is  a frame work for devolapinf=g aio app'),
 (np.float64(0.18538358765060423), 'Machin learning is a subset of AI')]